# GINO Evolution MLP Decoder With Super-Resolution Queries

Trains on the delta-mode evolution dataset with targets `[delta_state, u]`. The decoder can query `u` on arbitrary field coordinates for super-resolution style reconstruction.

In [2]:
RUN_TAG = "gino_evolution_mlp_sr"
RESULTS_SUBDIR = "result/gino_evolution_mlp_sr"
from __future__ import annotations

import inspect
import json
import math
import os
import random
import sys
import time
from collections import defaultdict
from datetime import datetime
from pathlib import Path
from typing import Dict, Optional, Sequence, Tuple

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    GNOBlock = None
    GNO_IMPORT_ERROR = error

SEED = int(os.environ.get("EVOLUTION_SR_SEED", "42"))
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def _repo_paths() -> Tuple[Path, Path]:
    cwd = Path.cwd().resolve()
    if (cwd / "FINAL").is_dir():
        return cwd, cwd / "FINAL"
    if cwd.name == "FINAL":
        return cwd.parent, cwd
    if cwd.parent.name == "FINAL":
        return cwd.parent.parent, cwd.parent
    return cwd, cwd / "FINAL"


def _int_or_none(raw: str) -> Optional[int]:
    raw = str(raw).strip().lower()
    if raw in {"", "none", "null", "all", "0"}:
        return None
    return int(raw)


def _csv_env(name: str, default: Sequence[str]) -> list[str]:
    raw = os.environ.get(name, "").strip()
    return [x.strip() for x in raw.split(",") if x.strip()] if raw else list(default)

REPO_ROOT, FINAL_DIR = _repo_paths()
DATASET_CANDIDATES = [
    FINAL_DIR / "processed_data" / "particle_evolution_dataset.npz",
]
DATASET_PATH = Path(os.environ.get("EVOLUTION_DATASET", "")).expanduser()
if str(DATASET_PATH) in {"", "."}:
    DATASET_PATH = next((p for p in DATASET_CANDIDATES if p.exists()), DATASET_CANDIDATES[0])
if not DATASET_PATH.is_absolute():
    DATASET_PATH = (Path.cwd() / DATASET_PATH).resolve()
if not DATASET_PATH.exists():
    raise FileNotFoundError(f"Missing particle_evolution_dataset.npz: {DATASET_PATH}")

RESULTS_DIR = FINAL_DIR / RESULTS_SUBDIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / f"{RUN_TAG}_best_model.pt"
HISTORY_PATH = RESULTS_DIR / f"{RUN_TAG}_history.json"

DEFAULT_INPUT_CHANNELS = [
    "Gamma_x", "Gamma_y", "Gamma_z", "sigma",
    "geom_dist", "geom_nx", "geom_ny", "geom_nz",
    "angle_of_attack", "phase",
]
CFG = {
    "seed": SEED,
    "run_tag": RUN_TAG,
    "epochs": int(os.environ.get("EVOLUTION_SR_EPOCHS", "60")),
    "lr": float(os.environ.get("EVOLUTION_SR_LR", "3e-4")),
    "weight_decay": float(os.environ.get("EVOLUTION_SR_WEIGHT_DECAY", "3e-5")),
    "eval_every": int(os.environ.get("EVOLUTION_SR_EVAL_EVERY", "2")),
    "batch_size": 1,
    "num_workers": int(os.environ.get("EVOLUTION_SR_NUM_WORKERS", "0")),
    "maximum_input_particles": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_INPUT_PARTICLES", "4096")),
    "maximum_train_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_TRAIN_QUERY_POINTS", "4096")),
    "maximum_eval_query_points": _int_or_none(os.environ.get("EVOLUTION_SR_MAX_EVAL_QUERY_POINTS", "12000")),
    "gradient_accumulation_steps": int(os.environ.get("EVOLUTION_SR_ACCUM_STEPS", "4")),
    "grad_clip_norm": float(os.environ.get("EVOLUTION_SR_GRAD_CLIP", "1.0")),
    "latent_res": int(os.environ.get("EVOLUTION_SR_LATENT_RES", "8")),
    "hidden_channels": int(os.environ.get("EVOLUTION_SR_HIDDEN", "96")),
    "gno_radius": float(os.environ.get("EVOLUTION_SR_GNO_RADIUS", "0.12")),
    "latent_mixer_layers": int(os.environ.get("EVOLUTION_SR_MIXER_LAYERS", "3")),
    "mlp_layers": int(os.environ.get("EVOLUTION_SR_MLP_LAYERS", "3")),
    "mlp_hidden": int(os.environ.get("EVOLUTION_SR_MLP_HIDDEN", "128")),
    "sr_grid_resolution": int(os.environ.get("EVOLUTION_SR_GRID_RES", "96")),
    "input_channels": _csv_env("EVOLUTION_SR_INPUT_CHANNELS", DEFAULT_INPUT_CHANNELS),
}

print("Dataset:", DATASET_PATH)
print("Results:", RESULTS_DIR)
print("Device :", DEVICE)
print(json.dumps(CFG, indent=2))


Dataset: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/processed_data/particle_evolution_dataset.npz
Results: /hpc/home/neerajc/FLOWUnsteady/Flow-reconstruction-in-VPM-using-FNO/FINAL/result/gino_evolution_mlp_sr
Device : cuda
{
  "seed": 42,
  "run_tag": "gino_evolution_mlp_sr",
  "epochs": 60,
  "lr": 0.0003,
  "weight_decay": 3e-05,
  "eval_every": 2,
  "batch_size": 1,
  "num_workers": 0,
  "maximum_input_particles": 4096,
  "maximum_train_query_points": 4096,
  "maximum_eval_query_points": 12000,
  "gradient_accumulation_steps": 4,
  "grad_clip_norm": 1.0,
  "latent_res": 8,
  "hidden_channels": 96,
  "gno_radius": 0.12,
  "latent_mixer_layers": 3,
  "mlp_layers": 3,
  "mlp_hidden": 128,
  "sr_grid_resolution": 96,
  "input_channels": [
    "Gamma_x",
    "Gamma_y",
    "Gamma_z",
    "sigma",
    "geom_dist",
    "geom_nx",
    "geom_ny",
    "geom_nz",
    "angle_of_attack",
    "phase"
  ]
}


In [ ]:
dataset_file = np.load(DATASET_PATH, allow_pickle=True)
feature_names_all = [str(x) for x in dataset_file["feature_names"].tolist()]
target_names = [str(x) for x in dataset_file["target_names"].tolist()]
if target_names[:7] != ["dx", "dy", "dz", "dGamma_x", "dGamma_y", "dGamma_z", "dsigma"]:
    raise RuntimeError(f"Expected delta-mode targets first; got {target_names}")
if target_names[7:10] not in (["u_x", "u_y", "u_z"], ["velocity_x", "velocity_y", "velocity_z"]):
    raise RuntimeError(f"Expected particle velocity u as target channels 7:10; got {target_names}")

missing_channels = [name for name in CFG["input_channels"] if name not in feature_names_all]
if missing_channels:
    raise KeyError(f"Missing input channels {missing_channels}; available={feature_names_all}")
active_input_feature_indices = [feature_names_all.index(name) for name in CFG["input_channels"]]
coord_feature_indices = [feature_names_all.index(name) for name in ("x", "y", "z")]
feature_names = [feature_names_all[i] for i in active_input_feature_indices]

frame_contexts = list(dataset_file["pair_contexts"] if "pair_contexts" in dataset_file.files else dataset_file["frame_contexts"])
frame_ranges = list(dataset_file["pair_ranges"] if "pair_ranges" in dataset_file.files else dataset_file["frame_ranges"])
inputs_t = np.asarray(dataset_file["inputs_t"], dtype=np.float32)
targets_delta = np.asarray(dataset_file["targets_delta"], dtype=np.float32)
inputs_t_norm = np.asarray(dataset_file["inputs_t_norm"], dtype=np.float32)
targets_delta_norm = np.asarray(dataset_file["targets_delta_norm"], dtype=np.float32)

inputs_by_pair = []
targets_by_pair = []
inputs_by_pair_norm = []
targets_by_pair_norm = []
for pair_range in frame_ranges:
    start = int(pair_range[3])
    end = int(pair_range[4])
    inputs_by_pair.append(inputs_t[start:end])
    targets_by_pair.append(targets_delta[start:end])
    inputs_by_pair_norm.append(inputs_t_norm[start:end])
    targets_by_pair_norm.append(targets_delta_norm[start:end])

train_pair_ids = np.asarray(dataset_file["train_pair_ids"], dtype=np.int64)
val_pair_ids = np.asarray(dataset_file["val_pair_ids"], dtype=np.int64) if "val_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
test_pair_ids = np.asarray(dataset_file["test_pair_ids"], dtype=np.int64) if "test_pair_ids" in dataset_file.files else np.asarray([], dtype=np.int64)
if "val_id_frame_ids" in dataset_file.files and len(dataset_file["val_id_frame_ids"]):
    val_pair_ids = np.asarray(dataset_file["val_id_frame_ids"], dtype=np.int64)
if len(val_pair_ids) == 0 and len(train_pair_ids) > 4:
    val_pair_ids = train_pair_ids[2::5]
    train_pair_ids = np.asarray([i for i in train_pair_ids if i not in set(val_pair_ids)], dtype=np.int64)

input_mean_all = np.asarray(dataset_file["in_mean"], dtype=np.float32).reshape(-1)
input_std_all = np.maximum(np.asarray(dataset_file["in_std"], dtype=np.float32).reshape(-1), 1e-8)
input_mean = input_mean_all[active_input_feature_indices]
input_std = input_std_all[active_input_feature_indices]
target_mean = np.asarray(dataset_file["out_mean"], dtype=np.float32).reshape(-1)
target_std = np.maximum(np.asarray(dataset_file["out_std"], dtype=np.float32).reshape(-1), 1e-8)
coord_min = np.asarray(dataset_file["coord_min"], dtype=np.float32).reshape(3) if "coord_min" in dataset_file.files else np.min(inputs_t[:, coord_feature_indices], axis=0)
coord_span = np.asarray(dataset_file["coord_span"], dtype=np.float32).reshape(3) if "coord_span" in dataset_file.files else np.ptp(inputs_t[:, coord_feature_indices], axis=0)
coord_span = np.maximum(coord_span, 1e-8)

delta_position_slice = slice(0, 3)
delta_strength_slice = slice(3, 7)
particle_velocity_slice = slice(7, 10)


def as_context(obj) -> Dict:
    if isinstance(obj, dict):
        return obj
    if hasattr(obj, "item"):
        item = obj.item()
        if isinstance(item, dict):
            return item
    return dict(obj)


def normalize_xyz(xyz: np.ndarray) -> np.ndarray:
    return np.clip((xyz.astype(np.float32) - coord_min[None, :]) / coord_span[None, :], 0.0, 1.0).astype(np.float32)


def sample_indices(n: int, cap: Optional[int], seed: int) -> np.ndarray:
    if cap is None or cap <= 0 or n <= int(cap):
        return np.arange(n, dtype=np.int64)
    rng = np.random.default_rng(int(seed))
    return np.sort(rng.choice(n, size=int(cap), replace=False)).astype(np.int64)

class EvolutionQueryDataset(Dataset):
    def __init__(self, pair_ids, split_name, max_input_particles, max_query_points):
        self.pair_ids = np.asarray(pair_ids, dtype=np.int64)
        self.split_name = split_name
        self.max_input_particles = max_input_particles
        self.max_query_points = max_query_points

    def __len__(self):
        return int(len(self.pair_ids))

    def __getitem__(self, index):
        pair_id = int(self.pair_ids[int(index)])
        features_all = np.asarray(inputs_by_pair[pair_id], dtype=np.float32)
        targets_all = np.asarray(targets_by_pair[pair_id], dtype=np.float32)
        n = min(features_all.shape[0], targets_all.shape[0])
        input_idx = sample_indices(n, self.max_input_particles, SEED + pair_id)
        query_local = sample_indices(len(input_idx), self.max_query_points, SEED + 100000 + pair_id)
        query_idx = input_idx[query_local]
        input_features = features_all[input_idx]
        query_features = features_all[query_idx]
        x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
        y = (targets_all[query_idx] - target_mean[None, :]) / target_std[None, :]
        context = as_context(frame_contexts[pair_id])
        return {
            "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])),
            "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)),
            "output_queries": torch.from_numpy(normalize_xyz(query_features[:, coord_feature_indices])),
            "query_xyz_raw": torch.from_numpy(query_features[:, coord_feature_indices].astype(np.float32)),
            "y": torch.from_numpy(np.nan_to_num(y).astype(np.float32)),
            "pair_id": torch.tensor(pair_id, dtype=torch.long),
            "case": str(context.get("case", "unknown")),
        }


def collate_one(batch):
    item = batch[0]
    out = {k: (v.unsqueeze(0) if torch.is_tensor(v) and k != "pair_id" else v) for k, v in item.items()}
    out["pair_id"] = item["pair_id"].view(1)
    return out

train_ds = EvolutionQueryDataset(train_pair_ids, "train", CFG["maximum_input_particles"], CFG["maximum_train_query_points"])
val_ds = EvolutionQueryDataset(val_pair_ids, "val", CFG["maximum_input_particles"], CFG["maximum_eval_query_points"])
test_ds = EvolutionQueryDataset(test_pair_ids, "test", CFG["maximum_input_particles"], CFG["maximum_eval_query_points"])
train_loader = DataLoader(train_ds, batch_size=1, shuffle=True, num_workers=CFG["num_workers"], collate_fn=collate_one)
val_loader = DataLoader(val_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0, collate_fn=collate_one)
print("Features:", feature_names)
print("Targets :", target_names)
print("Splits  :", {"train": len(train_ds), "val": len(val_ds), "test": len(test_ds)})


In [ ]:
def make_latent_queries(res: int, device: torch.device) -> torch.Tensor:
    line = torch.linspace(0.0, 1.0, int(res), dtype=torch.float32, device=device)
    xx, yy, zz = torch.meshgrid(line, line, line, indexing="ij")
    return torch.stack([xx, yy, zz], dim=-1).reshape(1, -1, 3)

LATENT_QUERIES = make_latent_queries(CFG["latent_res"], DEVICE)

def make_gnoblock(in_channels, out_channels, radius):
    if GNOBlock is None:
        raise RuntimeError("neuralop.layers.gno_block.GNOBlock is not available") from GNO_IMPORT_ERROR
    kwargs = dict(
        in_channels=in_channels,
        out_channels=out_channels,
        coord_dim=3,
        radius=float(radius),
        transform_type="linear",
        reduction="mean",
        pos_embedding_type="transformer",
        pos_embedding_channels=12,
        channel_mlp_layers=[out_channels, out_channels, out_channels],
    )
    accepted = set(inspect.signature(GNOBlock.__init__).parameters)
    if "use_torch_scatter_reduce" in accepted:
        kwargs["use_torch_scatter_reduce"] = False
    if "use_open3d_neighbor_search" in accepted:
        kwargs["use_open3d_neighbor_search"] = False
    return GNOBlock(**{k: v for k, v in kwargs.items() if k in accepted})

class LatentMLPDecoderGINO(nn.Module):
    def __init__(self, in_channels, out_channels, cfg):
        super().__init__()
        hidden = int(cfg["hidden_channels"])
        self.latent_res = int(cfg["latent_res"])
        self.lift = nn.Sequential(nn.Linear(in_channels, hidden), nn.GELU(), nn.Linear(hidden, hidden))
        self.encoder = make_gnoblock(hidden, hidden, cfg["gno_radius"])
        mixer = []
        for _ in range(max(int(cfg["latent_mixer_layers"]), 1)):
            mixer += [nn.Conv3d(hidden, hidden, kernel_size=3, padding=1), nn.GELU()]
        self.latent_mixer = nn.Sequential(*mixer)
        mlp = []
        width = int(cfg["mlp_hidden"])
        for i in range(max(int(cfg["mlp_layers"]), 1)):
            mlp += [nn.Linear(hidden + 3 if i == 0 else width, width), nn.GELU()]
        mlp.append(nn.Linear(width, out_channels))
        self.decoder = nn.Sequential(*mlp)

    def forward(self, input_geom, latent_queries, output_queries, x):
        outputs = []
        base_latent = latent_queries[0]
        r = self.latent_res
        for b in range(x.shape[0]):
            h = self.lift(x[b])
            latent = self.encoder(y=base_latent, x=input_geom[b], f_y=h)
            if latent.ndim == 3:
                latent = latent.squeeze(0)
            grid = latent.reshape(r, r, r, -1).permute(3, 0, 1, 2).unsqueeze(0)
            grid = self.latent_mixer(grid)
            q = output_queries[b].clamp(0.0, 1.0)
            sample_grid = (q * 2.0 - 1.0).view(1, -1, 1, 1, 3)
            sampled = torch.nn.functional.grid_sample(grid, sample_grid, align_corners=True, mode="bilinear")
            sampled = sampled.squeeze(0).squeeze(-1).squeeze(-1).transpose(0, 1)
            outputs.append(self.decoder(torch.cat([sampled, q], dim=-1)))
        return torch.stack(outputs, dim=0)

model = LatentMLPDecoderGINO(len(feature_names), len(target_names), CFG).to(DEVICE)
target_mean_t = torch.tensor(target_mean, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
target_std_t = torch.tensor(target_std, dtype=torch.float32, device=DEVICE).view(1, 1, -1)
print(model)
print("Trainable parameters:", f"{sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
def move_batch(batch):
    return {k: (v.to(DEVICE, non_blocking=True) if torch.is_tensor(v) else v) for k, v in batch.items()}

def predict(batch):
    return model(batch["input_geom"], LATENT_QUERIES, batch["output_queries"], batch["x"])

def denormalize_target(y):
    return y * target_std_t + target_mean_t

def relative_l2(pred, target, eps=1e-12):
    return torch.linalg.norm((pred - target).reshape(pred.shape[0], -1), dim=1) / torch.linalg.norm(target.reshape(target.shape[0], -1), dim=1).clamp_min(eps)

def grouped_loss(pred_norm, y_norm):
    pred = denormalize_target(pred_norm)
    y = denormalize_target(y_norm)
    mse = (pred - y) ** 2
    delta_xyz = mse[..., delta_position_slice].mean()
    delta_state = mse[..., delta_strength_slice].mean()
    velocity = mse[..., particle_velocity_slice].mean()
    return delta_xyz + delta_state + velocity, delta_xyz.detach(), delta_state.detach(), velocity.detach()

@torch.no_grad()
def evaluate(loader, max_batches=None):
    model.eval()
    if len(loader.dataset) == 0:
        return {"loss": math.nan, "rel_l2": math.nan, "u_rel_l2": math.nan, "batches": 0}
    losses, rels, u_rels = [], [], []
    for i, batch in enumerate(loader):
        if max_batches is not None and i >= int(max_batches):
            break
        batch = move_batch(batch)
        pred = predict(batch).float()
        loss, _, _, _ = grouped_loss(pred, batch["y"])
        pred_phys = denormalize_target(pred)
        y_phys = denormalize_target(batch["y"])
        losses.append(float(loss.item()))
        rels.append(float(relative_l2(pred_phys, y_phys).mean().item()))
        u_rels.append(float(relative_l2(pred_phys[..., particle_velocity_slice], y_phys[..., particle_velocity_slice]).mean().item()))
    return {"loss": float(np.mean(losses)), "rel_l2": float(np.mean(rels)), "u_rel_l2": float(np.mean(u_rels)), "batches": len(losses)}

optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(CFG["epochs"], 1))
history = []
best = float("inf")

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    losses, xyz_losses, state_losses, u_losses = [], [], [], []
    accum = max(int(CFG["gradient_accumulation_steps"]), 1)
    for step, batch in enumerate(train_loader, start=1):
        batch = move_batch(batch)
        pred = predict(batch)
        loss, xyz_loss, state_loss, u_loss = grouped_loss(pred, batch["y"])
        (loss / accum).backward()
        if step % accum == 0 or step == len(train_loader):
            nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip_norm"])
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
        losses.append(float(loss.item()))
        xyz_losses.append(float(xyz_loss.item()))
        state_losses.append(float(state_loss.item()))
        u_losses.append(float(u_loss.item()))
    scheduler.step()
    if epoch % CFG["eval_every"] == 0 or epoch == CFG["epochs"]:
        val = evaluate(val_loader, max_batches=CFG["maximum_eval_query_points"])
        test = evaluate(test_loader, max_batches=8)
        row = {"epoch": epoch, "train_loss": float(np.mean(losses)), "train_delta_xyz": float(np.mean(xyz_losses)), "train_delta_state": float(np.mean(state_losses)), "train_u": float(np.mean(u_losses)), "val": val, "test": test, "lr": optimizer.param_groups[0]["lr"]}
        history.append(row)
        print(json.dumps(row, indent=2))
        score = val["u_rel_l2"] if np.isfinite(val["u_rel_l2"]) else row["train_loss"]
        if score < best:
            best = score
            torch.save({
                "checkpoint_tag": RUN_TAG,
                "saved_at_utc": datetime.utcnow().isoformat() + "Z",
                "config": CFG,
                "model_state_dict": model.state_dict(),
                "feature_names_all": feature_names_all,
                "feature_names": feature_names,
                "target_names": target_names,
                "active_input_feature_indices": active_input_feature_indices,
                "coord_feature_indices": coord_feature_indices,
                "coord_min": coord_min,
                "coord_span": coord_span,
                "input_mean": input_mean,
                "input_std": input_std,
                "target_mean": target_mean,
                "target_std": target_std,
                "dataset_path": str(DATASET_PATH),
                "best_score": best,
                "history": history,
            }, CKPT_PATH)
            print("saved", CKPT_PATH)
HISTORY_PATH.write_text(json.dumps(history, indent=2))


In [ ]:
@torch.no_grad()
def predict_u_on_field_grid(pair_id: int, grid_resolution: Optional[int] = None, y_value: Optional[float] = None, max_input_particles: Optional[int] = None):
    """Query predicted particle-velocity u on an x-z plane across the full field.

    The model was supervised at particle positions, but the decoder accepts arbitrary
    normalized coordinates. This is the super-resolution query path.
    """
    model.eval()
    grid_resolution = int(grid_resolution or CFG["sr_grid_resolution"])
    features_all = np.asarray(inputs_by_pair[int(pair_id)], dtype=np.float32)
    input_idx = sample_indices(features_all.shape[0], max_input_particles or CFG["maximum_input_particles"], SEED + int(pair_id))
    input_features = features_all[input_idx]
    x = (input_features[:, active_input_feature_indices] - input_mean[None, :]) / input_std[None, :]
    x_line = np.linspace(coord_min[0], coord_min[0] + coord_span[0], grid_resolution, dtype=np.float32)
    z_line = np.linspace(coord_min[2], coord_min[2] + coord_span[2], grid_resolution, dtype=np.float32)
    xx, zz = np.meshgrid(x_line, z_line, indexing="xy")
    if y_value is None:
        y_value = float(np.median(features_all[:, coord_feature_indices[1]]))
    yy = np.full_like(xx, float(y_value), dtype=np.float32)
    query_xyz = np.stack([xx, yy, zz], axis=-1).reshape(-1, 3)
    batch = {
        "input_geom": torch.from_numpy(normalize_xyz(input_features[:, coord_feature_indices])).unsqueeze(0).to(DEVICE),
        "x": torch.from_numpy(np.clip(np.nan_to_num(x), -8.0, 8.0).astype(np.float32)).unsqueeze(0).to(DEVICE),
        "output_queries": torch.from_numpy(normalize_xyz(query_xyz)).unsqueeze(0).to(DEVICE),
    }
    pred_norm = predict(batch).float()
    pred_phys = denormalize_target(pred_norm).squeeze(0).cpu().numpy()
    u = pred_phys[:, particle_velocity_slice]
    return query_xyz, u, u.reshape(grid_resolution, grid_resolution, 3)

if len(test_ds) > 0:
    pair_id = int(test_ds.pair_ids[0])
elif len(val_ds) > 0:
    pair_id = int(val_ds.pair_ids[0])
else:
    pair_id = int(train_ds.pair_ids[0])
query_xyz, u_flat, u_grid = predict_u_on_field_grid(pair_id, grid_resolution=min(CFG["sr_grid_resolution"], 96))
u_mag = np.linalg.norm(u_grid, axis=-1)
fig, ax = plt.subplots(figsize=(7.2, 5.5), constrained_layout=True)
im = ax.imshow(u_mag, origin="lower", extent=[query_xyz[:,0].min(), query_xyz[:,0].max(), query_xyz[:,2].min(), query_xyz[:,2].max()], aspect="auto")
ax.set_xlabel("x")
ax.set_ylabel("z")
ax.set_title(f"SR field query |u|, pair_id={pair_id}")
plt.colorbar(im, ax=ax, label="|u|")
plot_path = RESULTS_DIR / f"{RUN_TAG}_sr_u_field.png"
fig.savefig(plot_path, dpi=220, bbox_inches="tight")
plt.show()
print("Saved:", plot_path)
